In [5]:
import os
os.chdir('..')

In [36]:
import re
import copy
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
def read_json_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

In [13]:
cluster_data = read_json_file('./data/bd_clustered_data.json')
address_data = read_json_file('./data/bangladesh_addresses.json')
df = pd.read_excel('./data/address_data_bn.xlsx')

In [15]:
df.head()

,Post Code,Division (BN),District (BN),Police Station (BN),Post Office (BN)
0,৮৭১০,বরিশাল,বরগুনা,আমতলী,আমতলী
1,৮৭১০,বরিশাল,বরগুনা,আমতলী,বাইনবুনিয়া
2,৮৭১০,বরিশাল,বরগুনা,আমতলী,চালাভাঙ্গা
3,৮৭১০,বরিশাল,বরগুনা,আমতলী,আমতলী
4,৮৭১০,বরিশাল,বরগুনা,আমতলী,বাইনবুনিয়া


In [21]:
cluster_data[0], cluster_data[-1]['post_offices'][:5]

({'district': 'কক্সবাজার',
  'thana': ['ঈদগাঁও',
   'উখিয়া',
   'কক্সবাজার সদর',
   'কুতুবদিয়া',
   'গোরকঘাট',
   'চকরিয়া',
   'চিরিঙ্গা',
   'টেকনাফ',
   'পেকুয়া',
   'মহেশখালী',
   'মাতামুহুরী',
   'মাতারবাড়ী',
   'রামু']},
 ['অচিন্তপুর', 'অন্নদানগর', 'অভয়নগর', 'অমরখানা', 'অমরপুর'])

In [27]:
address_data['add00001'], address_data['add00002']

({'add_id': 'add00001',
  'data': {'district': 'কক্সবাজার',
   'thana': 'উখিয়া',
   'post office': 'উখিয়া'}},
 {'add_id': 'add00002',
  'data': {'district': 'কক্সবাজার',
   'thana': 'উখিয়া',
   'post office': 'কুতুপালং'}})

In [29]:
bangla_count = 0
english_entries = []

for index, row in df.iterrows():
    for column, value in row.items():
        if column == 'Post Code':
            continue

        text = str(value).strip()

        has_bangla = any('\u0980' <= char <= '\u09FF' for char in text)
        has_english = any(('A' <= char <= 'Z') or ('a' <= char <= 'z') for char in text)

        if has_bangla:
            bangla_count += 1
        elif has_english:
            english_entries.append({
                'row': index,
                'column': column,
                'value': value
            })

print(f'Total data entries written in Bangla: {bangla_count}')
print(f'Total data entries written in English: {len(english_entries)}')

english_df = pd.DataFrame(english_entries)
english_df.head()

Total data entries written in Bangla: 41241
Total data entries written in English: 511


,row,column,value
0,84,Post Office (BN),Dokkhin Bhora
1,95,Post Office (BN),Kismat Choto Mokamia
2,127,Post Office (BN),Shatkor Betmor
3,146,Post Office (BN),Joberpar
4,153,Post Office (BN),Seral


In [34]:
# Remove rows containing English data
english_row_indices = english_df["row"].unique()

df = df.drop(index=english_row_indices).reset_index(drop=True)

print(f"Removed {len(english_row_indices)} rows.")
print(f"Remaining rows: {len(df)}")

Removed 511 rows.
Remaining rows: 9927


In [37]:
def normalize(value):
    return re.sub(r"\s+", " ", str(value).strip()).casefold()

def find_value(obj, aliases):
    aliases = {normalize(alias) for alias in aliases}

    if isinstance(obj, dict):
        for key, value in obj.items():
            if normalize(key) in aliases and not isinstance(value, (dict, list)):
                return value
            result = find_value(value, aliases)
            if result is not None:
                return result

    elif isinstance(obj, list):
        for item in obj:
            result = find_value(item, aliases)
            if result is not None:
                return result

    return None

def set_value(obj, aliases, value):
    aliases = {normalize(alias) for alias in aliases}
    updated = False

    if isinstance(obj, dict):
        for key in list(obj):
            if normalize(key) in aliases:
                obj[key] = value
                updated = True
            elif isinstance(obj[key], (dict, list)):
                updated |= set_value(obj[key], aliases, value)

    elif isinstance(obj, list):
        for item in obj:
            updated |= set_value(item, aliases, value)

    return updated

field_aliases = {
    "division": ["division", "division_bn"],
    "district": ["district", "district_bn"],
    "thana": ["thana", "police_station", "police station", "upazila"],
    "post_office": ["post_office", "post office", "post_office_bn"],
    "post_code": ["post_code", "post code", "postal_code", "postal code"],
}

def address_key(record):
    data = record.get("data", record)
    return tuple(
        normalize(find_value(data, field_aliases[field]) or "")
        for field in ["post_code", "division", "district", "thana", "post_office"]
    )

def add_address(values):
    global address_data

    numeric_ids = [
        int(re.search(r"\d+", key).group())
        for key in address_data
        if re.search(r"\d+", key)
    ]
    next_id = max(numeric_ids, default=0) + 1
    add_id = f"add{next_id:05d}"

    template = copy.deepcopy(next(iter(address_data.values())))
    template["add_id"] = add_id

    data = template.get("data", template)

    for field, value in values.items():
        aliases = field_aliases[field]

        if not set_value(data, aliases, value):
            # Fallback for schemas whose field names are not found.
            data[field] = value

    address_data[add_id] = template

# Existing address records
existing_addresses = {address_key(record) for record in address_data.values()}

# Add records found in df but missing from address_data
for _, row in df.iterrows():
    values = {
        "post_code": row.get("Post Code", ""),
        "division": row.get("Division (BN)", ""),
        "district": row.get("District (BN)", ""),
        "thana": row.get("Police Station (BN)", ""),
        "post_office": row.get("Post Office (BN)", ""),
    }

    key = tuple(normalize(values[field]) for field in values)

    if key not in existing_addresses:
        add_address(values)
        existing_addresses.add(key)

# Process cluster_data and add missing thanas
for district_entry in cluster_data:
    district = district_entry.get("district", "")

    for thana_entry in district_entry.get("thana", []):
        if isinstance(thana_entry, dict):
            thana = (
                thana_entry.get("thana")
                or thana_entry.get("name")
                or thana_entry.get("thana_name")
                or ""
            )
        else:
            thana = thana_entry

        thana_key = (
            normalize(""),
            normalize(""),
            normalize(district),
            normalize(thana),
            normalize(""),
        )

        has_thana = any(
            key[2] == thana_key[2] and key[3] == thana_key[3]
            for key in existing_addresses
        )

        if not has_thana:
            add_address({
                "post_code": "",
                "division": "",
                "district": district,
                "thana": thana,
                "post_office": "",
            })

            existing_addresses.add(thana_key)

# Find cluster post offices missing from the final address_data
final_post_offices = set()

for record in address_data.values():
    data = record.get("data", record)
    post_office = find_value(data, field_aliases["post_office"])

    if post_office:
        final_post_offices.add(normalize(post_office))

new_cluster_post_offices = []

for district_entry in cluster_data:
    for thana_entry in district_entry.get("thana", []):
        if not isinstance(thana_entry, dict):
            continue

        for post_office in thana_entry.get("post_offices", []):
            if isinstance(post_office, dict):
                post_office = (
                    post_office.get("name")
                    or post_office.get("post_office")
                    or post_office.get("post_office_name")
                )

            if post_office and normalize(post_office) not in final_post_offices:
                if post_office not in new_cluster_post_offices:
                    new_cluster_post_offices.append(post_office)

print(f"Final address records: {len(address_data)}")
print(f"New cluster post offices: {len(new_cluster_post_offices)}")

new_cluster_post_offices[:10]

Final address records: 10903
New cluster post offices: 0


[]

In [40]:
def is_empty(value):
    return value is None or str(value).strip() == ""

empty_address_records = []

for add_id, record in address_data.items():
    data = record.get("data", record)

    district_value = find_value(data, field_aliases["district"])
    thana_value = find_value(data, field_aliases["thana"])
    post_office_value = find_value(data, field_aliases["post_office"])

    empty_address_records.append({
        "add_id": add_id,
        "district": district_value,
        "thana": thana_value,
        "post_office": post_office_value,
        "district_empty": is_empty(district_value),
        "thana_empty": is_empty(thana_value),
        "post_office_empty": is_empty(post_office_value),
    })

empty_fields_df = pd.DataFrame(empty_address_records)

print(f"Total address records: {len(address_data)}")
print(f"Empty district fields: {empty_fields_df['district_empty'].sum()}")
print(f"Empty thana fields: {empty_fields_df['thana_empty'].sum()}")
print(f"Empty post office fields: {empty_fields_df['post_office_empty'].sum()}")

empty_address_df = empty_fields_df[
    empty_fields_df[["district_empty", "thana_empty", "post_office_empty"]].any(axis=1)
].reset_index(drop=True)

Total address records: 10903
Empty district fields: 0
Empty thana fields: 0
Empty post office fields: 138


In [46]:
empty_address_df.to_csv('./data/empty_address_records.csv', index=False, encoding='utf-8-sig')

In [47]:
filled_address_df = pd.read_csv('./data/filled_address_records.csv', encoding='utf-8-sig')

In [ ]:
# Fill empty post-office fields using filled_address_df
for _, row in filled_address_df.iterrows():
    add_id = row["add_id"]
    post_office = row["post_office"]

    if pd.isna(post_office) or not str(post_office).strip():
        continue

    if add_id in address_data:
        record = address_data[add_id]
        data = record.get("data", record)
        set_value(data, field_aliases["post_office"], str(post_office).strip())

# Remove records that still have an empty post-office field
address_data = {
    add_id: record
    for add_id, record in address_data.items()
    if not is_empty(
        find_value(record.get("data", record), field_aliases["post_office"])
    )
}

# Save the cleaned address data
output_path = "./data/bangladesh_addresses_filled.json"

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(address_data, file, ensure_ascii=False, indent=2)

print(f"Remaining records: {len(address_data)}")
print(f"Saved cleaned data to: {output_path}")